In [1]:
!pip install peft sentencepiece

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [19]:
import pandas as pd
import numpy as np
import torch
import random
import os
import re

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    matthews_corrcoef, balanced_accuracy_score,
    cohen_kappa_score, jaccard_score, hamming_loss,
    confusion_matrix
)
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling
)
from peft import get_peft_model, LoraConfig, TaskType
from tqdm import tqdm


In [2]:
# Gendered dataset
df = pd.read_csv("data/combined_letters_gendered.csv")[["full_text", "label"]].dropna()
df["label_text"] = df["label"].map({1: "male", 0: "female"})

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df["full_text"],
    df["label_text"],
    test_size=0.2,
    stratify=df["label_text"],
)

In [4]:
def format_prompt(text, label=None):
    prompt = (
        f"Based on the content of the following letter of recommendation, determine if the applicant is male or female:\n"
        f"\"{text}\"\nAnswer:"
    )
    return prompt if label is None else f"{prompt} {label}"

In [5]:
train_prompts = [format_prompt(txt, label) for txt, label in zip(X_train, y_train)]
test_prompts = [format_prompt(txt) for txt in X_test]

In [6]:
# model_name = "meta-llama/Llama-2-7b-hf"
model_name = "NousResearch/Llama-2-7b-chat-hf"
peft_model_path = "models/llama2_gendered"

In [7]:
# Tokenizer
if os.path.exists(peft_model_path):
    tokenizer = AutoTokenizer.from_pretrained(peft_model_path)
else:
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

tokenizer.pad_token = tokenizer.eos_token

In [8]:
# Model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
# LORA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(base_model, lora_config)

In [10]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [11]:
# Prepare datasets
train_dataset = Dataset.from_dict({"text": train_prompts})
test_dataset = Dataset.from_dict({"text": test_prompts, "label": y_test.tolist()})

tokenized_train = train_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_test = test_dataset.map(tokenize, batched=True).remove_columns(["text"])

Map:   0%|          | 0/7189 [00:00<?, ? examples/s]

Map:   0%|          | 0/1798 [00:00<?, ? examples/s]

In [12]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [13]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results_llama2_gendered",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="no",
    logging_steps=10,
    report_to=None,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    tokenizer=tokenizer,
    data_collator=data_collator
)


/tmp/ipykernel_931475/3009314395.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Training
trainer.train()

Step,Training Loss
10,2.491300
20,2.195700
30,2.201900
40,2.006900
50,2.023300
60,1.993300
70,1.978600
80,1.930400
90,1.997300
100,1.939300


In [ ]:
model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

In [14]:
# Check for adapter files
adapter_model_file = os.path.join(peft_model_path, "adapter_model.bin")
adapter_config_file = os.path.join(peft_model_path, "adapter_config.json")

# Load PEFT adapter if it exists, else just use base model
if os.path.exists(adapter_model_file) and os.path.exists(adapter_config_file):
    model = PeftModel.from_pretrained(base_model, peft_model_path)
else:
    model = base_model

In [20]:
# Inference
def classify(prompt, max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only what's after 'Answer:' and clean up
    answer = decoded.split("Answer:")[-1].strip().lower()

    # Optional: Just grab first word (in case it adds reasoning)
    answer = re.split(r'[\s\.,:;\n]', answer)[0]

    # Return only 'male' or 'female' if matched
    if "male" in answer:
        return "male"
    elif "female" in answer:
        return "female"
    else:
        return "unknown"

predictions = [classify(p) for p in tqdm(test_prompts)]


  8%|▊         | 152/1798 [00:38<06:57,  3.94it/s]


 17%|█▋        | 303/1798 [01:17<06:18,  3.95it/s]


 25%|██▌       | 455/1798 [01:56<04:49,  4.64it/s]


 34%|███▎      | 606/1798 [02:34<05:04,  3.92it/s]


 42%|████▏     | 757/1798 [03:13<04:36,  3.77it/s]


...

 67%|██████▋   | 1207/1798 [05:08<02:28,  3.97it/s]


 75%|███████▌  | 1356/1798 [05:46<01:51,  3.98it/s]


 84%|████████▎ | 1504/1798 [06:23<01:15,  3.90it/s]


 92%|█████████▏| 1653/1798 [07:01<00:36,  3.96it/s]


100%|██████████| 1798/1798 [07:38<00:00,  3.92it/s]


In [21]:
# Evaluation
def compute_metrics(true, pred):
    pred_clean = [p if p in {"male", "female"} else "unknown" for p in pred]
    true_clean = [t.lower() for t in true]

    acc = accuracy_score(true_clean, pred_clean)
    precision, recall, f1, _ = precision_recall_fscore_support(true_clean, pred_clean, average="macro", zero_division=0)
    mcc = matthews_corrcoef(true_clean, pred_clean)
    bal_acc = balanced_accuracy_score(true_clean, pred_clean)
    kappa = cohen_kappa_score(true_clean, pred_clean)
    jaccard = jaccard_score(true_clean, pred_clean, average="macro", labels=["male", "female"])
    hamming = hamming_loss(true_clean, pred_clean)
    cm = confusion_matrix(true_clean, pred_clean, labels=["male", "female", "unknown"])

    print("Accuracy:", acc)
    print("F1 Score:", f1)
    print("Confusion Matrix:\n", cm)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
        "cohen_kappa": kappa,
        "jaccard": jaccard,
        "hamming_loss": hamming
    }

metrics = compute_metrics(y_test.tolist(), predictions)


Accuracy: 0.3231368186874305
F1 Score: 0.176381299332119
Confusion Matrix:
 [[581   0 660]
 [374   0 183]
 [  0   0   0]]


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2446: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [22]:
predictions

['male',
 'unknown',
 'male',
 'male',
 'unknown',
 'male',
 'unknown',
 'unknown',
 'unknown',
 'unknown',
 'male',
 'male',
 'male',
 'unknown',
 'male',
 'unknown',
 'male',
 'unknown',
 'male',
 'male',
 'unknown',
 'male',
 'male',
 'male',
 'unknown',
 'unknown',
 'male',
 'unknown',
 'male',
 'unknown',
 'male',
 'male',
 'unknown',
 'male',
 'male',
 'male',
 'male',
 'male',
 'unknown',
 'male',
 'unknown',
 'unknown',
 'unknown',
 'unknown',
 'unknown',
 'male',
 'male',
 'male',
 'unknown',
 'unknown',
 'unknown',
 'male',
 'unknown',
 'unknown',
 'male',
 'male',
 'male',
 'unknown',
 'unknown',
 'unknown',
 'unknown',
 'unknown',
 'male',
 'male',
 'male',
 'unknown',
 'male',
 'male',
 'male',
 'male',
 'male',
 'male',
 'unknown',
 'male',
 'male',
 'unknown',
 'male',
 'unknown',
 'male',
 'male',
 'male',
 'male',
 'unknown',
 'unknown',
 'unknown',
 'unknown',
 'unknown',
 'male',
 'unknown',
 'male',
 'unknown',
 'unknown',
 'unknown',
 'male',
 'unknown',
 'unknown'